# Chit 4 baseline: a camera trap that wakes up for animals · **MCU**
**AIMS Edge ML for Science · Capstone baseline** · 🟩 AI-on

Train a small CNN on 96 × 96 **grayscale** images (animal vs. empty), measure animal recall when at most **5%** of the empties are kept, in fp32 and int8, and count the cost on a microcontroller.

This is a **starting point, not a good design**: a small, simple model that runs end to end (data → training → the chit's metric → its cost on the target),
so your team has something to beat. Every number it prints is yours to improve. Run time: ~3 min on a laptop CPU (measured), plus the first download; training uses the GPU automatically if the runtime has one.

> **Before you start:** **File → Save a copy in Drive**, then run the cells in order.

In [ ]:
# --- Colab setup: run once per session (~1 min) ---
import os, sys, glob, subprocess
IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = False       # True: keep the chit's data on your Google Drive between sessions
LAB = "capstone/baselines"
if IN_COLAB:
    ROOT = "/content/aims-fastml-labs"
    if not os.path.exists(ROOT):
        bundles = glob.glob("/content/*bundle*.zip")
        if bundles:
            subprocess.run(["unzip", "-q", "-o", bundles[0], "-d", ROOT], check=True)
        else:
            subprocess.run(["git", "clone", "-q", "https://github.com/SagarA17/aims-fastml-labs.git", ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *["onnx", "onnxruntime"]], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["FASTML_DATA"] = "/content/drive/MyDrive/fastml_data"
    os.chdir(f"{ROOT}/{LAB}")
print("working directory:", os.getcwd())

In [ ]:
import os, sys, time, json, warnings
sys.path.append("../.."); warnings.filterwarnings("ignore")
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from common.budget import profile, fits
from capstone.baselines.tools import FAST, fit, predict, to_onnx, ort_session, ort_int8, ort_predict, ort_latency, mcu_latency, energy_uj, report
torch.manual_seed(0); np.random.seed(0)
os.makedirs("my_models", exist_ok=True)
T0 = time.time()

## 0. Understand the data
Before any model, look at the data: what one sample is, what the classes look like, how the splits were made, and why the chit uses its metric.
This part only reads and plots (≲ 1 min). The model starts in Part 1.

### 0.1 The task in plain words
A camera trap is a camera in a box, tied to a post in the open countryside. A motion or heat sensor triggers it, and it takes a short burst of photos.
In Karoo National Park most bursts are **empty**: wind moves the grass, the sun heats a rock. Volunteers then look at every photo by hand.
A tiny classifier on the camera's microcontroller could throw away the empty photos before they are stored, saving battery, memory card space and volunteer time.
**One decision of the model** = for one photo, "keep it (there may be an animal)" or "discard it (empty)".

### 0.2 What one sample is
One sample is one photo, **3 × 96 × 96** unsigned 8-bit integers: three colour channels (red, green, blue), 96 rows, 96 columns; 0 = dark, 255 = full brightness.
The camera's 2592 × 2000 photo was cropped to the central square and shrunk to 96 × 96, so small or distant animals become a few pixels.
The camera's black information strip (date, time, temperature) stays in, at the bottom.
**Label:** 1 = animal, 0 = empty. Labels were given by volunteers **per sequence** (per burst), so every photo of a burst gets the burst's label.
**Sequence:** the burst a photo belongs to (`sequences()`), e.g. `KAR_S1#B03#1#462` = season 1, camera site B03, roll 1, sequence 462.
There is no "night" label; we mark a photo as **night** when its lower part is much brighter than its upper part (the camera's flash lights the ground under a black sky).
This is a rule of thumb that we checked by eye. The timestamps in the original metadata do not help: some camera clocks are wrong (site B03 shows daylight at "2 a.m.").

In [ ]:
import matplotlib.pyplot as plt
from capstone.data.karoo import load, sequences
X0, y0 = load("val")                              # 96 × 96 RGB, as stored (the baseline below uses grayscale)
seq0 = sequences("val")
print("one image:", X0.shape[1:], X0.dtype, "= (colour R, G, B; rows; columns), values 0-255 | label 1 = animal, 0 = empty")
print("first image: label", y0[0], "| sequence", seq0[0], "| top-left pixel (R, G, B):", X0[0, :, 0, 0])
gray = X0.astype(np.float32).mean(1)                              # (n, 96, 96) brightness
bright = gray.mean((1, 2))
NIGHT = gray[:, 60:88].mean((1, 2)) - gray[:, :24].mean((1, 2)) > 100   # flash photo: lit ground (lower part) under a black sky (upper part)
print(f"night (flash) images by this rule: {NIGHT.mean():.1%} of val")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for k, c, name in ((1, "C3", "animal"), (0, "C0", "empty")):
    ax[0].hist(gray[y0 == k][:400].ravel(), np.arange(0, 257, 4), density=True, histtype="step", color=c, label=f"{name} (all pixels of 400 images)")
    for night, ls in ((False, "-"), (True, "--")):
        sel = (y0 == k) & (NIGHT == night)
        ax[1].hist(bright[sel], np.arange(0, 257, 8), density=True, histtype="step", color=c, ls=ls,
                   label=f"{name}, {'night' if night else 'day'} ({sel.sum():,} images)")
ax[0].set(xlabel="pixel brightness (mean of R, G, B; 0 = black, 255 = white)", ylabel="density", title="Pixel values")
ax[1].set(xlabel="mean brightness of the image", ylabel="density", title="One number per image: brightness")
for a in ax:
    a.legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"images that are almost white (mean brightness > 220): {np.mean(bright > 220):.1%}")

**What to notice.** Left: the pixel values are spread over the whole range; the spike at 0 is the black information strip, and the spike at 255 is
over-exposed (pure white) sky or flash-lit ground, which is more common in animal photos. Right: the mean brightness of a photo is a weak clue. Empty daytime photos are
a little darker on average than animal photos, and night (flash) photos are brighter than day photos, but all the curves overlap widely.
Brightness alone cannot separate animals from empties.

### 0.3 Look at it
Random photos from `val`: animals by day, animals at night, empties by day, empties at night. The title gives the camera site.

In [ ]:
rng = np.random.default_rng(0)
pick = lambda sel, n: rng.choice(np.where(sel)[0], min(n, int(sel.sum())), replace=False)
rows = [("animal, day", pick((y0 == 1) & ~NIGHT, 8)), ("animal, night", pick((y0 == 1) & NIGHT, 8)),
        ("empty, day", pick((y0 == 0) & ~NIGHT, 8)), ("empty, night", pick((y0 == 0) & NIGHT, 8))]
fig, ax = plt.subplots(4, 8, figsize=(16, 8.6))
for r, (name, idx) in enumerate(rows):
    for a, i in zip(ax[r], idx):
        a.imshow(X0[i].transpose(1, 2, 0)); a.set_xticks([]); a.set_yticks([])
        a.set_title(f"{name} · site {seq0[i].split('#')[1]}", fontsize=8)
plt.tight_layout(); plt.show()

**What to notice.** Night photos are flash photos in colour (not infrared grey-scale images), with a black sky and a bright, often over-exposed foreground.
Some animals fill the frame, but in several "animal" photos the animal is tiny, hidden, or not visible at all at 96 × 96: the label belongs to the whole burst, and the animal
may be in another photo of it. Empty photos of one site look almost the same from burst to burst (the camera does not move).
So the model must notice small changes in a scene it has seen many times, which is hard at this resolution.

In [ ]:
ids, counts = np.unique(seq0, return_counts=True)
burst = [ids[(counts == 3)][j] for j in np.where(y0[[np.where(seq0 == s)[0][0] for s in ids[counts == 3]]] == 1)[0][:2]]
fig, ax = plt.subplots(1, 6, figsize=(16, 3))
for j, s in enumerate(burst):
    for t, i in enumerate(np.where(seq0 == s)[0]):
        a = ax[3 * j + t]; a.imshow(X0[i].transpose(1, 2, 0)); a.set_xticks([]); a.set_yticks([])
        a.set_title(f"sequence {s.split('#')[1]}#{s.split('#')[3]}: image {t + 1} of 3", fontsize=8)
plt.tight_layout(); plt.show()
print("images per sequence in val:", {int(c): int(n) for c, n in zip(*np.unique(counts, return_counts=True))}, "(size: number of sequences)")

Two animal sequences of three photos each. In the first, an eland fills the frame; in the second (also an eland), the animal is a few pixels behind the bush and barely visible.
(The loader keeps the photos of a burst together but not their order in time.) Photos of one burst are near-duplicates, which is why the splits are made
by sequence, never by photo: otherwise the test set would contain near-copies of training photos.

### 0.4 The numbers that matter

In [ ]:
print("split   images  sequences  animal  night (flash)")
for split in ("train", "val", "test"):
    X, y = (X0, y0) if split == "val" else load(split)
    g = X.astype(np.float32).mean(1); n = g[:, 60:88].mean((1, 2)) - g[:, :24].mean((1, 2)) > 100
    print(f"{split:<6} {len(y):>7,} {len(np.unique(sequences(split))):>10,}  {y.mean():6.1%}  {n.mean():6.1%}")
    if split == "train":
        site_tr = np.array([s.split("#")[1] for s in sequences("train")]); y_tr0 = y
    del X, g
print("\nanimal share per camera site (train):")
print("  ".join(f"{s} {np.mean(y_tr0[site_tr == s]):.0%}" for s in np.unique(site_tr)))
u, c = np.unique(site_tr[y_tr0 == 0], return_counts=True); top4 = np.argsort(-c)[:4]
print(f"{c[top4].sum() / c.sum():.0%} of the empty training photos come from 4 sites:", ", ".join(u[top4]))

**Classes:** about 35% animal in every split. This is a subset: every animal sequence, plus a random sample of empty sequences.
In the full data set only about 16% of the photos have an animal (83% are empty); the metric below does not depend on this mix.
**Splits:** by **sequence**: the sequences are shuffled once (fixed seed) and split 70 / 15 / 15, separately for animal and empty sequences.
The **same 15 camera sites** appear in every split, which is realistic (a camera stays where it is), but note the per-site table: at some sites almost every photo is an animal
(E03: 98%), at others almost every photo is empty (F02: 2%), because most empty photos come from four sites (printed above). A model can learn "which site is this?"
instead of "is there an animal?", and that shortcut would not work at a new site.
**Value range:** integers 0–255; the baseline divides by 255 and converts to grayscale. Colour and resolution are your choice (`load(size=..., grayscale=...)`).

### 0.5 The metric, explained on the data
The camera keeps a photo when the model's score is above a threshold. The threshold is set so that only **5% of the empty photos** are kept
(they are the cost: battery, storage, volunteer time), and the metric is the share of **animal photos that are kept** at that threshold (recall).
Accuracy would mislead: in the wild 83% of photos are empty, so "discard everything" is 83% accurate and useless. Because the threshold is set on the empty photos and the recall
is measured on the animal photos, the metric is the same whatever the animal/empty mix of the data set.

In [ ]:
del X0, gray

## 1. Data
The loader stores 96 × 96 RGB; this baseline asks for grayscale (`grayscale=True`). Colour and resolution are levers you can change.

In [ ]:
from capstone.data.karoo import load, recall_at_empties_kept
kw = dict(grayscale=True, size=96)
X_tr, y_tr = load("train", max_items=2_000 if FAST else None, **kw)
X_va, y_va = load("val", **kw); X_te, y_te = load("test", **kw)
for name, y in (("train", y_tr), ("val", y_va), ("test", y_te)):
    print(f"{name:<5} {len(y):>6,} images, {np.mean(y):.1%} animal")
XTR, XVA, XTE = [X.astype(np.float32) / 255 for X in (X_tr, X_va, X_te)]

## 2. The model, costed before training
Four strided 3 × 3 convolutions (96 → 6 pixels), global average pooling, a linear layer.

In [ ]:
model = nn.Sequential(nn.Conv2d(1, 8, 3, stride=2, padding=1), nn.ReLU(),
                      nn.Conv2d(8, 16, 3, stride=2, padding=1), nn.ReLU(),
                      nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
                      nn.Conv2d(32, 32, 3, stride=2, padding=1), nn.ReLU(),
                      nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, 2))
p = profile(model, (1, 96, 96))
print(f"params {p['params']:,} | MACs {p['macs']:,} | int8 weights {p['weight_bytes']:,} B | peak act {p['peak_act_bytes']:,} B")
TARGET = dict(max_weight_bytes=1024 * 1024, max_act_bytes=256 * 1024, mac_rate=100e6, max_latency_s=1.0)   # DRAFT values from chits.md
fits(p, TARGET);

## 3. Train (class-weighted) and measure

In [ ]:
w = len(y_tr) / (2 * np.bincount(y_tr, minlength=2))
fit(model, XTR, y_tr, epochs=1 if FAST else 12, bs=64, lr=3e-3, class_weight=w)
prob = torch.softmax(torch.as_tensor(predict(model, XTE)), 1)[:, 1].numpy()
REC32 = recall_at_empties_kept(prob, y_te)
print(f"test animal recall with 5% of empties kept, fp32: {REC32:.4f}")

## 4. int8 (weights and activations), with ONNX Runtime static quantization

In [ ]:
to_onnx(model.cpu(), XTR[:1], "my_models/karoo_fp32.onnx")
ort_int8("my_models/karoo_fp32.onnx", XTR[np.random.default_rng(0).permutation(len(XTR))[:512]], "my_models/karoo_int8.onnx")
sess8 = ort_session("my_models/karoo_int8.onnx")
OUT8 = ort_predict(sess8, XTE)

P8 = torch.softmax(torch.as_tensor(OUT8), 1)[:, 1].numpy()
REC8 = recall_at_empties_kept(P8, y_te)
print(f"test animal recall with 5% of empties kept, int8: {REC8:.4f}   <- the chit's metric")
print()
print("other operating points   empties kept   recall fp32   recall int8")
for k in (0.20, 0.10, 0.05, 0.02, 0.01):
    print(f"{'':<24} {k:>12.0%} {recall_at_empties_kept(prob, y_te, kept=k):>13.3f} {recall_at_empties_kept(P8, y_te, kept=k):>13.3f}")

In [ ]:
ROW = report("4 karoo", (len(XTR), len(XVA), len(XTE)), (1, 96, 96), f"recall@5%empties int8 {REC8:.4f} (fp32 {REC32:.4f}); @20% {recall_at_empties_kept(P8, y_te, kept=0.20):.3f}, @10% {recall_at_empties_kept(P8, y_te, kept=0.10):.3f}, @2% {recall_at_empties_kept(P8, y_te, kept=0.02):.3f}", p,
             f"MCU counted: {1e3 * mcu_latency(p['macs'], 100e6):.1f} ms at 100 M MAC/s",
             "CDLA-Permissive 1.0 (LILA, confirmed)", extra=dict(minutes=round((time.time() - T0) / 60, 1)))

---
**Where next.** Read your chit card and `targets.md` again with these numbers in front of you: which budget binds first? The levers are listed in `targets.md`.